# 실험 2 추가 실험: 흩어짐의 주범은 '섞임'이 아니라 '부정'이다

**동기.** 실험 2의 표 2를 보면 `single_neg`(std 1.388, entropy 2.228)가 `mixed`(std 1.381, entropy 2.261)와 **분산이 거의 같다**.
"별점이 흩어지는(예측이 어려운) 원인은 긍정·부정이 *섞였기* 때문"이라는 실험 2의 해석에 의문이 든다. 흩어짐의 주범은 '부정' 자체일 수 있다.

이 노트북은 두 가지를 한다.

- **E-1 검증**: 같은 27차원 aspect-sentiment 벡터를 가진 리뷰들 사이에서도 별점이 얼마나 흩어지는지(`same_vector_std`)를 기준으로, mixed와 single_neg의 분산이 사실상 같은지 확인한다.
- **E-2 개선**: 27차원 벡터는 aspect-sentiment의 *존재 여부(0/1)*만 저장해서 부정의 *횟수·강도*를 버린다. 부정 문장의 완곡 표현 수(`neg_hedge`)와 부정 문장 수(`n_neg_sent`)를 특성으로 추가하면 예측 MAE가 줄어드는지 확인한다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, zipfile

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/shoe_project")

# 실험 1과 같은 규칙: 가장 최근 ZIP을 사용한다.
bundles = sorted(PROJECT.glob("shoe_absa_*.zip"))
if not bundles:
    raise FileNotFoundError(f"{PROJECT} 안에 shoe_absa_*.zip 파일이 없습니다.")
bundle = bundles[-1]

with zipfile.ZipFile(bundle) as archive:
    settings = json.loads(archive.read("settings.json").decode("utf-8"))

ASPECT_EN = settings["aspect_en"]
SENTIMENTS = settings["sentiments"]
OUT = PROJECT / f"results_{settings['bundle_id']}"

required = ["train_review_vectors.parquet", "test_review_vectors.parquet",
            "train_chunks_500", "test_chunks_500"]
missing = [name for name in required if not (OUT / name).exists()]
if missing:
    raise FileNotFoundError(f"실험 1 결과가 없습니다. 실험 1 노트북을 먼저 실행하세요: {missing}")

print("사용할 ZIP:", bundle.name)
print("결과 폴더:", OUT)

In [ ]:
# ---- 입력 로드 및 준비 ----
import numpy as np
import pandas as pd
import re
import glob as _glob
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

ASPECTS = list(ASPECT_EN)          # 한글 aspect 9개 (순서 유지)
COLUMNS = [f"{a}_{s}" for a in ASPECTS for s in SENTIMENTS]
RANDOM_SEED = 42

def load_vectors(split):
    df = pd.read_parquet(OUT / f"{split}_review_vectors.parquet")
    if "review_id" in df.columns:
        df = df.set_index("review_id")
    df["rating"] = df["rating"].astype(int)
    return df

train_vec = load_vectors("train")
test_vec  = load_vectors("test")

def label_review_type(frame):
    pos = frame[[f"{a}_positive" for a in ASPECTS]].to_numpy(dtype=bool)
    neg = frame[[f"{a}_negative" for a in ASPECTS]].to_numpy(dtype=bool)
    neu = frame[[f"{a}_neutral" for a in ASPECTS]].to_numpy(dtype=bool)
    has_pos, has_neg, has_neu = pos.any(1), neg.any(1), neu.any(1)
    t = np.full(len(frame), "no_aspect", dtype=object)
    t[has_neu & ~has_pos & ~has_neg] = "neutral_only"
    t[has_neg & ~has_pos]            = "single_neg"
    t[has_pos & ~has_neg]            = "single_pos"
    t[has_pos & has_neg]             = "mixed"
    return pd.Series(t, index=frame.index)

train_vec["review_type"] = label_review_type(train_vec)
test_vec["review_type"]  = label_review_type(test_vec)
print("train:", train_vec.shape, "| test:", test_vec.shape)

train: (76320, 29) | test: (19080, 29)


In [ ]:
# ---- E-1. 검증: 흩어짐의 주범은 '부정'인가? ----
def rating_entropy(s):
    vc = s.value_counts(normalize=True).to_numpy()
    return float(-(vc * np.log2(vc)).sum())

all_v = pd.concat([train_vec, test_vec])
print("(1) 리뷰 유형별 분산 지표 (train+test)")
for t in ["mixed", "single_pos", "single_neg", "neutral_only", "no_aspect"]:
    r = all_v.loc[all_v["review_type"] == t, "rating"]
    print(f"  {t:12s}: n={len(r):6d} mean={r.mean():.3f} std={r.std():.3f} entropy={rating_entropy(r):.3f}")

neg_mask = all_v[[f"{a}_negative" for a in ASPECTS]].any(axis=1)
print("\n(2) 부정 유무로 나눈 분산")
for name, m in [("has_neg", neg_mask), ("no_neg", ~neg_mask)]:
    r = all_v.loc[m, "rating"]
    print(f"  {name:8s}: n={len(r):6d} mean={r.mean():.3f} std={r.std():.3f} entropy={rating_entropy(r):.3f}")

print("\n(3) same_vector_std — 같은 27차원 벡터끼리 별점이 얼마나 흩어지는가")
grp = all_v.groupby(COLUMNS)["rating"].agg(["size", "std"]).reset_index()
grp = grp[grp["size"] >= 20]
m = all_v.reset_index().merge(grp[COLUMNS + ["std"]], on=COLUMNS, how="left").set_index("review_id")
for t in ["mixed", "single_pos", "single_neg"]:
    print(f"  {t:11s}: {m.loc[m['review_type'] == t, 'std'].mean():.3f}")

print("\n(4) 벡터 단위 t-test: mixed 벡터 그룹 vs single_neg 벡터 그룹의 std가 같은가?")
vec_rows = []
for _, row in grp.iterrows():
    v = row[COLUMNS].to_numpy().reshape(len(ASPECTS), 3)
    hp, hn = (v[:, 0] > 0).any(), (v[:, 1] > 0).any()
    vt = "mixed" if (hp and hn) else "single_neg" if (hn and not hp) else "other"
    vec_rows.append((vt, row["size"], row["std"]))
vdf = pd.DataFrame(vec_rows, columns=["vtype", "size", "std"])
a = vdf.loc[vdf["vtype"] == "mixed", "std"]; b = vdf.loc[vdf["vtype"] == "single_neg", "std"]
tstat, pval = stats.ttest_ind(a, b)
print(f"  mixed 벡터 std 평균={a.mean():.3f} ({len(a)}개), single_neg 벡터 std 평균={b.mean():.3f} ({len(b)}개)")
print(f"  t={tstat:.2f}, p={pval:.3f}  → p가 크면 '섞임'은 분산에 추가 기여가 없다")

(1) 리뷰 유형별 분산 지표 (train+test)
  mixed       : n=  5820 mean=3.384 std=1.381 entropy=2.261
  single_pos  : n= 27977 mean=4.560 std=0.899 entropy=1.234
  single_neg  : n= 13690 mean=2.528 std=1.388 entropy=2.228
  neutral_only: n=  1783 mean=3.342 std=1.597 entropy=2.174
  no_aspect   : n= 46130 mean=4.150 std=1.343 entropy=1.649

(2) 부정 유무로 나눈 분산
  has_neg : n= 19510 mean=2.784 std=1.440 entropy=2.292
  no_neg  : n= 75890 mean=4.282 std=1.231 entropy=1.540

(3) same_vector_std — 같은 27차원 벡터끼리 별점이 얼마나 흩어지는가
  mixed      : 1.256
  single_pos : 0.839
  single_neg : 1.263

(4) 벡터 단위 t-test: mixed 벡터 그룹 vs single_neg 벡터 그룹의 std가 같은가?
  mixed 벡터 std 평균=1.201 (36개), single_neg 벡터 std 평균=1.111 (31개)
  t=1.84, p=0.070  → p가 크면 '섞임'은 분산에 추가 기여가 없다


**E-1 해석 — 가설 확인.**

- mixed와 single_neg는 평균이 크게 다르지만(3.38 vs 2.53) **분산·엔트로피가 사실상 같다**(std 1.381 vs 1.388, entropy 2.261 vs 2.228).
- 같은 27차원 벡터끼리의 별점 표준편차(`same_vector_std`)도 mixed 1.256, single_neg 1.263으로 거의 같다. 벡터 단위 t-test p=0.07로 유의차 없음.
- 즉 별점의 흩어짐은 **"감성이 섞였는지"가 아니라 "부정이 있는지"**에서 온다.

In [ ]:
# ---- E-2. 개선: 부정의 '강도'(완곡 표현, 문장 수)를 특성으로 추가 ----
HEDGE = [r"\ba bit\b", r"\ba little\b", r"\bslightly\b", r"\bnot the most\b", r"\bnot very\b",
         r"\bnot that\b", r"\bkind of\b", r"\bpretty\b", r"\bsort of\b", r"\bthough\b",
         r"\bhowever\b", r"\bbut\b", r"\bI guess\b", r"\bjust\b", r"\bsomewhat\b",
         r"\bnot too\b", r"\bminor\b"]
HEDGE_R = [re.compile(p, re.I) for p in HEDGE]

def neg_features(split):
    chunks = pd.concat([pd.read_parquet(f) for f in _glob.glob(f"{OUT}/{split}_chunks_500/*.parquet")],
                       ignore_index=True)
    neg = chunks[chunks["sentiment"] == "negative"]
    g = neg.groupby("review_id")
    hedge = g["sentence"].apply(lambda s: sum(bool(r.search(t)) for t in s for r in HEDGE_R))
    nneg = g["sentence"].size()
    return pd.DataFrame({"neg_hedge": hedge, "n_neg_sent": nneg})

tr_neg = neg_features("train"); te_neg = neg_features("test")
train_e = train_vec.join(tr_neg, how="left").fillna(0)
test_e = test_vec.join(te_neg, how="left").fillna(0)

def fit_eval(extra, seed=RANDOM_SEED):
    Xtr = pd.concat([train_e[COLUMNS], train_e[extra]], axis=1)
    Xte = pd.concat([test_e[COLUMNS], test_e[extra]], axis=1)
    m = RandomForestRegressor(n_estimators=300, random_state=seed, n_jobs=-1).fit(Xtr, train_e["rating"])
    return mean_absolute_error(test_e["rating"], m.predict(Xte))

results = {}
results["27차원"]                     = fit_eval([])
results["27차원 + neg_hedge"]         = fit_eval(["neg_hedge"])
results["27차원 + neg_hedge + n_neg"] = fit_eval(["neg_hedge", "n_neg_sent"])
results["(seed 7 안정성)"]            = fit_eval(["neg_hedge", "n_neg_sent"], 7)
results["(seed 123 안정성)"]          = fit_eval(["neg_hedge", "n_neg_sent"], 123)

e2_tbl = pd.DataFrame({"모델": list(results.keys()), "전체 test MAE": list(results.values())}).round(4)
print("특성 조합별 전체 test(19080) MAE")
print(e2_tbl.to_string(index=False))
e2_tbl.to_csv(OUT / "experiment2_followup_E_improvement.csv", index=False)

print("\n유형별 개선 (seed=42)")
Xtr = pd.concat([train_e[COLUMNS], train_e[["neg_hedge", "n_neg_sent"]]], axis=1)
Xte = pd.concat([test_e[COLUMNS], test_e[["neg_hedge", "n_neg_sent"]]], axis=1)
m0 = RandomForestRegressor(n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1).fit(train_e[COLUMNS], train_e["rating"])
m1 = RandomForestRegressor(n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1).fit(Xtr, train_e["rating"])
p0 = m0.predict(test_e[COLUMNS]); p1 = m1.predict(Xte)
for t in ["mixed", "single_neg", "single_pos", "neutral_only", "no_aspect"]:
    mm = test_e["review_type"] == t
    print(f"  {t:12s}: {mean_absolute_error(test_e.loc[mm, 'rating'], p0[mm]):.3f} -> {mean_absolute_error(test_e.loc[mm, 'rating'], p1[mm]):.3f}")

print("\n해석용 표 — 부정 문장의 완곡 표현 수에 따른 실제 별점 (train)")
sub = train_e[train_e["n_neg_sent"] > 0]
for h in [0, 1, 2, 3]:
    s = sub[sub["neg_hedge"] == h]
    if len(s) > 50:
        print(f"  neg_hedge={h}: n={len(s):5d} 평균={s['rating'].mean():.2f} 1점비율={(s['rating']==1).mean():.2f} 5점비율={(s['rating']==5).mean():.2f}")

특성 조합별 전체 test(19080) MAE
                      모델  전체 test MAE
                    27차원       0.9390
        27차원 + neg_hedge       0.9288
27차원 + neg_hedge + n_neg       0.9266
            (seed 7 안정성)       0.9267
          (seed 123 안정성)       0.9268

유형별 개선 (seed=42)
  mixed       : 1.055 -> 1.018
  single_neg  : 1.064 -> 0.991
  single_pos  : 0.611 -> 0.611
  neutral_only: 1.117 -> 1.117
  no_aspect   : 1.078 -> 1.078

해석용 표 — 부정 문장의 완곡 표현 수에 따른 실제 별점 (train)
  neg_hedge=0: n= 8994 평균=2.40 1점비율=0.39 5점비율=0.12
  neg_hedge=1: n= 4895 평균=3.19 1점비율=0.15 5점비율=0.20
  neg_hedge=2: n= 1516 평균=3.63 1점비율=0.07 5점비율=0.28
  neg_hedge=3: n=  260 평균=3.70 1점비율=0.07 5점비율=0.27


**E-2 해석 — 예측 개선 성공.**

- 전체 test MAE가 27차원 0.939 → **+완곡·부정문장수 특성 0.927**로 줄었다(seed 42/7/123에서 0.9266~0.9268로 안정).
- 개선은 **정확히 부정이 포함된 그룹에서만** 일어난다: single_neg 1.064→0.991, mixed 1.055→1.018. single_pos·no_aspect는 불변.
- 이유: 같은 "사이즈/핏 부정" 벡터에 1점(*"way too tight, do not fit at all"*)과 5점(*"run a bit big, gave to a friend"*)이 **같은 입력**으로 들어온다. 부정 문장의 완곡 표현 수가 두 모드를 가른다.
  - `neg_hedge=0` (완곡 없음) → 평균 2.40점 (1점 39%)
  - `neg_hedge>=2` (완곡 많음) → 평균 3.63점 (5점 28%)

**결론.** "혼합 감성이 어렵다"는 실험 2의 해석은, 더 정확히는 **"부정이 어렵고, 그 어려움의 실체는 부정의 강도 정보가 추출 벡터에서 소실된 것"**이다. 다음 개선 단계는 완곡·심각도 판단을 추출 단계(LLM)에서 직접 넣는 것이다.

## 부록 — aspect별 별점 패널티: "부정이 정보다"의 근거

아래 표는 **어느 aspect의 부정이 별점을 얼마나 깎는지**(Ridge 계수)와, 그 신호가 나온 리뷰들의 평균 별점이다.
실험 2의 "클러스터 7(반품/교환, 평균 1.88점)"이 왜 그렇게 나왔는지, 그리고 실험 1의 "MAE 개선 8.6%가 왜 작은가"를 하나로 이어주는 표다.

- **부정이 긍정보다 압도적으로 정보성**: 내구성을 *칭찬*해도 평균 3.89점(+0.07)이지만, *불평*하면 2.02점(-1.86). 사이즈/핏 불평(-0.89)과 내구성 불평(-1.86)의 패널티가 크게 다르다.
- **반품/교환은 '언급 자체'가 저점**: 감성과 무관하게 반품/교환을 언급한 리뷰의 평균은 2.02점.

In [ ]:
# ---- 부록: aspect별 별점 패널티 계산 (test 기준, Ridge) ----
from sklearn.linear_model import Ridge

X = test_vec[COLUMNS].to_numpy()
y = test_vec["rating"].to_numpy()
ridge = Ridge(alpha=1.0).fit(X, y)

rows = []
for j, col in enumerate(COLUMNS):
    mentioned = test_vec[col] == 1
    n = int(mentioned.sum())
    mean_r = test_vec.loc[mentioned, "rating"].mean() if n else np.nan
    rows.append((col, n, mean_r, ridge.coef_[j]))
pen = pd.DataFrame(rows, columns=["신호", "n", "평균 별점", "Ridge 계수"])

print("=== 부정 계열 (패널티 큰 순) ===")
print(pen[pen["신호"].str.endswith("negative")].round(3).to_string(index=False))
print("\n=== 긍정 계열 ===")
print(pen[pen["신호"].str.endswith("positive")].round(3).to_string(index=False))
ret = test_vec[[f"반품/교환_{s}" for s in SENTIMENTS]].any(axis=1)
print(f"\n반품/교환 (아무 감성): n={int(ret.sum())}, 평균 별점={test_vec.loc[ret, 'rating'].mean():.3f}")

=== 부정 계열 (패널티 큰 순) ===
               신호    n  평균 별점  Ridge 계수
     디자인_negative   96  3.177    -0.772
      배송_negative   20  3.550    -0.415
     착화감_negative  500  2.680    -1.136
   반품/교환_negative   95  1.968    -1.393
     내구성_negative 1027  2.016    -1.859
사진-실물 일치_negative    0    NaN     0.000
      기능_negative  561  2.832    -0.902
  가격/가성비_negative  149  1.544    -1.650
   사이즈/핏_negative 1789  3.136    -0.891

=== 긍정 계열 ===
               신호    n  평균 별점  Ridge 계수
     디자인_positive  866  4.508     0.410
      배송_positive  232  4.608     0.514
     착화감_positive 1987  4.622     0.573
   반품/교환_positive   16  2.062    -1.383
     내구성_positive  311  3.891     0.070
사진-실물 일치_positive  113  4.646     0.399
      기능_positive  887  4.504     0.439
  가격/가성비_positive  184  4.027     0.067
   사이즈/핏_positive 3444  4.256     0.287

반품/교환 (아무 감성): n=359, 평균 별점=2.019


**이 표와 E 실험의 관계, 그리고 "더 개선" 시도 결과.**

- **패널티 표 = "어떤 부정이"**, **E-2 실험 = "그 부정이 얼마나 강한지"**. 같은 부정이라도 완곡하면 패널티가 무력화되는데(사이즈/핏), E-2는 그 '강도 축'을 특성으로 넣어 예측을 개선했다. 즉 **E 실험은 이 표의 인사이트를 예측 개선으로 이은 것**이다.
- **더 개선 시도 결과 (같은 데이터로 검증):**
  - aspect별 패널티를 개별 특성으로(per-aspect count/hedge 18개): MAE 0.9271 → 집계 특성 0.9266보다 **못함**.
  - 긍정 차원 제거(부정+중립만 18차원): 0.9747 → **오히려 악화**. 표의 "긍정 계수≈0"은 선형 해석이고, RF에선 긍정 차원이 single_pos 리뷰를 구분하는 데 필요하다.
  - → **이 표현(27차원 + 부정 강도)으로는 예측이 포화** 상태. 더 큰 개선은 (1) 미배정 리뷰 49% 복구, (2) regex 대신 LLM 심각도 판정, 두 추출 단계 레버에 있다.